# Soil Survey Metadata Discovery

Query and filter Soil Data Access (SDA) survey area metadata with `soildb`.

Topics:
- Load the national survey area catalog (`get_sacatalog`)
- Query map units by survey area (`get_mapunit_by_areasymbol`)
- Filter survey areas by keyword and state code


## 1. Imports


In [1]:
import time

# Import soildb components
from soildb import (
    get_mapunit_by_areasymbol,
    get_sacatalog,
)

## 2. Load Survey Area Catalog

Load the national survey catalog from SDA. Convenience functions manage client connections automatically.


In [9]:
# Query the national survey area catalog
response = await get_sacatalog()

# Convert to pandas DataFrame
df_catalog = response.to_pandas()

print(f"Loaded {len(df_catalog)} survey areas")
print("\nFirst few surveys:")
print(df_catalog[["areasymbol", "areaname"]].head())

Loaded 3380 survey areas

First few surveys:
  areasymbol                               areaname
0      AK600  Matanuska-Susitna Valley Area, Alaska
1      AK605                 Anchorage Area, Alaska
2      AK610         Greater Fairbanks Area, Alaska
3      AK612              Copper River Area, Alaska
4      AK615             Gerstle River Area, Alaska


In [3]:
# Query map units for Boone County, Iowa (IA015)
response = await get_mapunit_by_areasymbol("IA015")
df_mapunits = response.to_pandas()

print(f"Found {len(df_mapunits)} map units in IA015")
print("\nFirst few map units:")
print(df_mapunits.head())

Found 101 map units in IA015

First few map units:
     mukey musym                                             muname  \
0  2835023   107           Webster clay loam, 0 to 2 percent slopes   
1  2550233  1135  Coland clay loam, 0 to 2 percent slopes, frequ...   
2  2550234   135  Coland clay loam, 0 to 2 percent slopes, occas...   
3  2765538  138B                Clarion loam, 2 to 6 percent slopes   
4  2550236  138C               Clarion loam, 6 to 10 percent slopes   

         mukind  muacres areasymbol            areaname  
0  Consociation     9724      IA015  Boone County, Iowa  
1  Consociation     3865      IA015  Boone County, Iowa  
2  Consociation     6564      IA015  Boone County, Iowa  
3  Consociation    13600      IA015  Boone County, Iowa  
4  Consociation     4075      IA015  Boone County, Iowa  


## 3. Filter Survey Catalog Records

Filter catalog records by name matching.


In [4]:
# Filter survey areas matching keywords
surveys_with_keyword = df_catalog[
    df_catalog["areaname"].str.contains("County|State", case=False, na=False)
].head(20)

metadata_list = []
for _idx, row in surveys_with_keyword.iterrows():
    metadata_list.append(
        {
            "areasymbol": row["areasymbol"],
            "areaname": row["areaname"],
            "saversion": row["saversion"],
        }
    )

print(f"Loaded {len(metadata_list)} survey records")
print("\nSample surveys:")
for m in metadata_list[:5]:
    print(f"  - {m['areasymbol']}: {m['areaname']}")

Loaded 20 survey records

Sample surveys:
  - AK774: Kenai Mountains and Kachemak Bay State Park Area, Alaska
  - AK794: Kachemak Bay State Wilderness Park Area, Alaska
  - AL001: Autauga County, Alabama
  - AL003: Baldwin County, Alabama
  - AL005: Barbour County, Alabama


## 4. Filter Survey Areas by Keyword

Search survey names with pandas string methods.


In [5]:
# Search survey names for specific keywords
keyword_surveys = {}
for keyword in ["iowa", "minnesota", "county"]:
    matching = df_catalog[
        df_catalog["areaname"].str.contains(keyword, case=False, na=False)
    ]
    keyword_surveys[keyword] = matching["areasymbol"].tolist()

for keyword, surveys in keyword_surveys.items():
    print(f"'{keyword}': {len(surveys)} surveys")
    if surveys:
        print(f"  Examples: {surveys[:3]}")

'iowa': 103 surveys
  Examples: ['CO061', 'IA001', 'IA003']
'minnesota': 92 surveys
  Examples: ['MN001', 'MN003', 'MN005']
'county': 2669 surveys
  Examples: ['AL001', 'AL003', 'AL005']


## 5. Filter Survey Areas by State Code

Filter survey areas by state prefix.


In [6]:
# Filter survey symbols by two-letter state prefix
state_codes = ["IA", "IL", "MO", "MN"]
state_surveys = {}

for state in state_codes:
    matching = df_catalog[df_catalog["areasymbol"].str.startswith(state, na=False)]
    state_surveys[state] = matching["areasymbol"].tolist()

for state, surveys in state_surveys.items():
    print(f"{state}: {len(surveys)} surveys")
    if surveys:
        display = surveys[:3] + (["..."] if len(surveys) > 3 else [])
        print(f"  {', '.join(display)}")

IA: 99 surveys
  IA001, IA003, IA005, ...
IL: 102 surveys
  IL001, IL003, IL005, ...
MO: 114 surveys
  MO001, MO003, MO005, ...
MN: 92 surveys
  MN001, MN003, MN005, ...


## 6. Survey Area Summary Statistics

Count survey areas across states.


In [7]:
# Summarize survey counts by state
total_surveys = len(df_catalog)
unique_states = df_catalog["areasymbol"].str[:2].nunique()

print(f"Total surveys loaded: {total_surveys}")
print(f"Unique states/territories: {unique_states}")

print("\nSurvey count by state (top 15):")
state_counts = df_catalog["areasymbol"].str[:2].value_counts().head(15)
for state, count in state_counts.items():
    print(f"  {state}: {count} surveys")

Total surveys loaded: 3380
Unique states/territories: 61

Survey count by state (top 15):
  TX: 232 surveys
  AK: 137 surveys
  CA: 120 surveys
  MO: 114 surveys
  VA: 108 surveys
  KS: 105 surveys
  IL: 102 surveys
  NC: 100 surveys
  IA: 99 surveys
  GA: 95 surveys
  TN: 94 surveys
  NE: 93 surveys
  IN: 92 surveys
  MN: 92 surveys
  OH: 88 surveys


## 7. Inspect Survey Records

Inspect individual survey attributes from the catalog.


In [8]:
# Inspect individual survey attributes
if metadata_list:
    sample_areasymbol = metadata_list[0]["areasymbol"]
    sample_areaname = metadata_list[0]["areaname"]

    print(f"Survey: {sample_areasymbol} - {sample_areaname}")
    print(f"saversion: {metadata_list[0]['saversion']}")

    print("\nAccessing survey data multiple times:")
    access_times = []
    for _i in range(5):
        start = time.time()
        _ = metadata_list[0]["areaname"]
        elapsed = time.time() - start
        access_times.append(elapsed)

    avg_time = sum(access_times) / len(access_times)
    print(f"Average access time: {avg_time * 1000000:.2f} microseconds")
    print("(5 accesses, all equally fast due to caching)")
else:
    print("No metadata available")

Survey: AK774 - Kenai Mountains and Kachemak Bay State Park Area, Alaska
saversion: 7

Accessing survey data multiple times:
Average access time: 0.43 microseconds
(5 accesses, all equally fast due to caching)


## 8. Summary

- `get_sacatalog()` retrieves the national soil survey area catalog.
- `get_mapunit_by_areasymbol()` retrieves map units for a specific survey area.
- Standard pandas methods filter survey areas by name, keyword, or state code.
